# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# str.lower().str.contains('pop')]
pop_tarts = cat[cat['product_name'].str.lower().str.contains('pop')]

print(f"number of pop tart like products = {len(pop_tarts)}")
pop_tarts

number of pop tart like products = 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO

# w/ strawberry
strawberry_pop_tarts = pop_tarts[pop_tarts['product_name'].str.lower().str.contains('strawberry')]
print("strawberry pop tart products:")
display(strawberry_pop_tarts[['sku', 'product_name']])

# w/out strawberry (~ = not)
non_strawberry_pop_tarts = pop_tarts[~pop_tarts['product_name'].str.lower().str.contains('strawberry')]
print("\nnon strawberry pop tart products:")
display(non_strawberry_pop_tarts[['sku', 'product_name']])

strawberry pop tart products:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry



non strawberry pop tart products:


,sku,product_name


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
sku_map = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW'
}

cat['canonical_sku'] = cat['sku'].replace(sku_map)
cat

,sku,product_name,category,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,POPTART-STRAW


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print("cats before cleaning:")
print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = cat['category'].str.strip().str.title()

print("\ncats after cleaning")
print(cat['category_clean'].value_counts(dropna=False))

cats before cleaning:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

cats after cleaning
category_clean
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import re

def normalize_name(name):
    if not isinstance(name, str):
        return ""

    # cleaning up
    name_clean = name.lower()
    name_clean = re.sub(r'[^a-z0-9\s]', ' ', name_clean)
    words = sorted(name_clean.split())
    return " ".join(words)

#normalized name column
cat['normalized_name'] = cat['product_name'].apply(normalize_name)
groups = cat.groupby('normalized_name').filter(lambda x: x['sku'].nunique() > 1)
groups.sort_values(by='normalized_name')[['sku', 'product_name', 'normalized_name']]

,sku,product_name,normalized_name
0,PT-12,pop-tarts strawberry,pop strawberry tarts
2,POPTART-STRAW,Pop-Tarts Strawberry,pop strawberry tarts


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['canonical_sku'].notna().all(), "null values in canonical_sku"
original_row_count = 3  # make sure didnt drop rows
assert len(cat) == original_row_count, "row count changed"

print("mapping is safe")

mapping is safe


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

### Write-up Answers

a)
Besides poptarts, coca cola product families and bounty paper towels are also fragmented. you can see different entries like coke vs. coca-cola, different naming conventions for packs /sizing. we can group these by mapping the variations to "COCA-COLA-CLASSIC" sku. Same with the paper towels, there are different names for strenghs (like double roll, 2-ply). These could be grouped by taking the ply type and roll counts into structured attributes and mapping them under the same parent family ID as well.


b)  Consolidation becomes more of a business deicison when the differences in product descriptions would actually impact variations in retailing/sales and represent different consumer needs. So for example, the products regular coke vs. zero sugar coke would appeal to different consumers (people who are watching their sugar intake, diabetics, people dieting, etc.),  so grouping these variants together would impact invventory tracking and hurt the business model

For this current catalogue, we have 'PT-12' and 'SKU#459812' which are both strawverry poptarts, but we can't be certain if they are the exact same packaging with 8 packs, or if one of them has more/less servings. this could potentially result in an issue with inventory management so we would likely need to ask the the buyer for the breakfast aisle,or someone in charge of inventory database administration to check if these separate SKUs map to the same barcode or UPC.